In [ ]:
!curl -L -o fda-drug-adverse-event-reports-2015-to-2026-faers.zip\
https://www.kaggle.com/api/v1/datasets/download/kanchana1990/fda-drug-adverse-event-reports-2015-to-2026-faers

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
csv_path = 'fda_adverse_events_2015_2026_CLEAN.csv' 
# Read CSV file
df = pd.read_csv(csv_path)

In [3]:
df.head()

,report_id,receive_date,year,month,quarter,serious,serious_flags,is_fatal,is_hospitalized,is_life_threat,...,manufacturer,pharm_class,num_drugs,drug_count_category,patient_age_years,age_group,patient_sex,patient_weight_kg,country,report_age_days
0,10004718,2015-02-11,2015,2,2015Q1,Yes,Hospitalization,False,True,False,...,Glaxosmithkline Llc,Unknown,7,Polypharmacy(6+),64.0,Middle-Aged(41-65),Female,NaN,US,4063
1,10004926,2015-02-13,2015,2,2015Q1,Yes,Hospitalization,False,True,False,...,Bayer Healthcare Pharmaceuticals Inc.,Progestin [EPC]; Progestin-containing Intraute...,1,Single,20.0,Adult(19-40),Female,54.0,US,4061
2,10005223,2015-02-19,2015,2,2015Q1,Yes,Hospitalization,False,True,False,...,Cordavis Limited; Abbvie Inc.,Unknown,14,Polypharmacy(6+),60.0,Middle-Aged(41-65),Female,NaN,US,4055
3,10005378,2015-02-17,2015,2,2015Q1,Yes,Hospitalization,False,True,False,...,Unknown,Unknown,34,Polypharmacy(6+),20.0,Adult(19-40),Female,NaN,BR,4057
4,10005980,2015-02-21,2015,2,2015Q1,Yes,NaN,False,False,False,...,Live Betr Llc; American Sales Company; Little ...,Unknown,7,Polypharmacy(6+),NaN,Unknown,Female,NaN,GB,4053


In [4]:
selected_cols = [
    "serious",
    "is_fatal",
    "is_hospitalized",
    "is_life_threat",
    "is_disabling",
    "reactions",
    "primary_reaction",
    "reaction_outcomes",
    "suspect_drug"
]

df = df[selected_cols]

In [ ]:
df = df.dropna(subset=[
    "serious",
    "reactions",
    "reaction_outcomes"
])

df_sample = df.sample(n=2000, random_state=42)


In [ ]:
train_df, temp_df = train_test_split(
    df_sample,
    test_size=0.2,
    random_state=42,
    stratify=df_sample["serious"]  # keeps label balance
)

# Second: Split Temp into Eval (10%) and Test (10%)
eval_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["serious"]
)

# -------------------------------
# 5. Check sizes
# -------------------------------
print("Train size:", len(train_df))
print("Eval size:", len(eval_df))
print("Test size:", len(test_df))

# -------------------------------
# 6. Save datasets
# -------------------------------
train_df.to_csv("train.csv", index=False)
eval_df.to_csv("eval.csv", index=False)
test_df.to_csv("test.csv", index=False)


def build_case_text(row):
    return f"""
    Adverse Event Report:

    A patient reported the following adverse reactions:
    {row['reactions']}

    Primary reaction:
    {row['primary_reaction']}

    Outcome of the event:
    {row['reaction_outcomes']}

    Suspected drug:
    {row['suspect_drug']}

    Patient details:
    Age: {row.get('patient_age_years', 'Unknown')} years
    Sex: {row.get('patient_sex', 'Unknown')}
    """



results = []

for _, row in df.iterrows():
    case_text = build_case_text(row)
    
    prompt = f"""
You are an expert FDA adverse event reviewer.

Analyze the case and produce a structured clinical review.

STRICT RULES:
- Use only the information provided
- Do NOT hallucinate facts
- Follow FDA seriousness criteria

Known structured signals:
- Serious: {row['serious']}
- Fatal: {row['is_fatal']}
- Hospitalized: {row['is_hospitalized']}
- Life Threatening: {row['is_life_threat']}
- Disabling: {row['is_disabling']}

Output JSON:
{{
  "seriousness": "Yes/No/Uncertain",
  "seriousness_criteria": [],
  "reasoning": "",
  "evidence_spans": [],
  "primary_meddra_term": "",
  "secondary_meddra_terms": [],
  "causality": "",
  "confidence": 0.0
}}

Case:
{case_text}
"""
    response = call_llm(prompt)
    parsed = parse_output(response)

    if parsed:
        results.append(parsed)




prompt += f"""

Known structured signals:
- Serious: {row['serious']}
- Fatal: {row['is_fatal']}
- Hospitalized: {row['is_hospitalized']}
- Life Threatening: {row['is_life_threat']}
- Disabling: {row['is_disabling']}

Ensure your output is consistent with these indicators.
"""


final_data = []

for _, row in df.iterrows():
    case_text = build_case_text(row)

    full_prompt = prompt.format(case_text=case_text)

    response = call_llm(full_prompt)
    parsed = parse_output(response)

    if parsed:
        final_data.append({
            "input_text": case_text,
            "serious_label": row["serious"],  # original truth
            "llm_output": parsed
        })
